# AI Fairness Dashboard

This notebook **showcases** the end-to-end pipeline without replacing the production scripts. Run it after `run_pipeline.py` (or with existing `artifacts/`) so tables and figures load from saved outputs.

**Setup:** open this folder from the repo root, kernel using the project venv (`./venv`), and set the working directory to the repository root (see first code cell).

## 1. What we built

- **Deterministic audit:** standard classifiers → fairness metrics (Python / AIF360) → rule-based severity, causes, mitigations → Semantic Scholar evidence.
- **LLM benchmark (Gemini / OpenAI):** same Python-computed metrics + evidence → qualitative audit and refinement cycles → scored against a shared reference spec.
- **Gold-standard direction:** a project-defined rubric (metrics + narrative + evidence + mitigations); our pipeline is one system evaluated against it, alongside LLMs.

In [ ]:
from pathlib import Path
import json
import pandas as pd

# Repo root: notebooks/ -> parent is repo
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
ART = REPO / "artifacts"
assert ART.is_dir(), f"Expected {ART}. Run pipeline from repo root or cd to repo root."
print("Repo:", REPO)

## 2. Pipeline flow (chronological)

1. Clean → train (`RandomForest` / `LogisticRegression` per dataset scripts) → `classification_predictions.csv`  
2. `compute_fairness.py` → `fairness_metrics.csv`  
3. `qualitative_analysis.py` → `qualitative_report.md` + `qualitative_research_evidence.json`  
4. Optional: `llm_fairness_analysis.py` / `openai_fairness_analysis.py` → provider folders under `metrics/fairness/` (mirrored in `artifacts/`)

Orchestrator: `run_pipeline.py` at repo root.

## 3. Deterministic fairness metrics (German Credit & HMDA)

Loaded from consolidated artifacts after a full run.

In [ ]:
from IPython.display import display

for ds in ["german_credit", "hmda"]:
    p = ART / ds / "fairness" / "fairness_metrics.csv"
    if not p.exists():
        print(f"Missing {p}")
        continue
    print(f"\n=== {ds} ===")
    display(pd.read_csv(p))

## 4. Semantic Scholar evidence (sample)

Queries are built in `scripts/scholarly_evidence.py` from dataset name, protected attributes, and deterministic causes/mitigations. Papers prefer `citation_count >= 5` when possible, with fallback for newer work.

In [ ]:
ev_path = ART / "german_credit" / "fairness" / "qualitative_research_evidence.json"
if ev_path.exists():
    data = json.loads(ev_path.read_text(encoding="utf-8"))
    for attr, papers in list(data.items())[:1]:
        print(f"Attribute: {attr}")
        for i, paper in enumerate(papers[:3], 1):
            print(f"  {i}. {paper.get('title', '')[:80]}...")
            print(f"     citations={paper.get('citation_count')}, meets_threshold={paper.get('meets_citation_threshold', 'n/a')}")
else:
    print("Run qualitative step to generate", ev_path)

## 5. LLM benchmark snapshot (OpenAI)

Napkin math and final scores from saved artifacts (no API call in this cell).

In [ ]:
rows = []
for ds, label in [("german_credit", "German Credit"), ("hmda", "HMDA")]:
    nap = ART / ds / "fairness" / "openai" / "llm_napkin_math.json"
    raw = ART / ds / "fairness" / "openai" / "llm_raw_response.json"
    if not nap.exists() or not raw.exists():
        continue
    u = json.loads(nap.read_text(encoding="utf-8")).get("usage", {})
    r = json.loads(raw.read_text(encoding="utf-8"))
    cycles = r.get("cycles", [])
    final = cycles[-1]["score"]["total_score"] if cycles else None
    rows.append({
        "dataset": label,
        "openai_cost_usd": u.get("total_cost_usd"),
        "openai_final_score": final,
    })
from IPython.display import display

display(pd.DataFrame(rows) if rows else pd.DataFrame({"note": ["No OpenAI artifacts; run llm_benchmark step"]}))

## 6. Visuals (from `artifacts/visualizations/`)

OpenAI benchmark plots and any consolidated charts produced by the pipeline.

In [ ]:
from IPython.display import Image, display

viz = REPO / "artifacts" / "visualizations"
for name in [
    "openai_cycle_scores.png",
    "openai_severity_agreement.png",
    "openai_cost_latency.png",
    "severity_comparison.png",
]:
    p = viz / name
    if p.exists():
        display(Image(filename=str(p)))

## 7. Deep dive: read full reports

- Deterministic: `artifacts/{dataset}/fairness/qualitative_report.md`
- OpenAI: `artifacts/{dataset}/fairness/openai/llm_fairness_report.md` and `benchmark_comparison.md`
- Comparison PDF: `artifacts/consolidated/deterministic_vs_openai_short_report.pdf`

## 8. Re-run pipeline (optional)

From repo root, in a terminal:

```bash
./venv/bin/python run_pipeline.py --datasets german_credit hmda
```

Then re-run this notebook to refresh tables and figures.